In [ ]:
# Imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, interact_manual, widgets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split    
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc
from math import pi
import tqdm
from collections import Counter
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, LSTM,GlobalAveragePooling1D, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
# Import the necessary layers
from tensorflow.keras.layers import Conv1D, GlobalAveragePooling1D, Dropout,MaxPooling1D
from tensorflow.keras import Model

In [ ]:
settings_DF=pd.read_csv('/Volumes/FBK-NAS/01_Tool_WearDatasets/00_NASA_Milling/settings.csv')
# the ID column of settings_DF is unnamed, so we rename it
#settings_DF.drop(columns={'Unnamed: 0'}, inplace=True)
# load signals and merge them, as they have all the same shape
signal_DFs=[]
for signal in ['vib_table', 'vib_spindle', 'AE_table', 'AE_spindle']:
    signal_DFs.append(pd.read_csv('/Volumes/FBK-NAS/01_Tool_WearDatasets/00_NASA_Milling/'+signal+'.csv'))
    print(signal_DFs[-1].shape)
# merge the signals
merged_DF=pd.concat(signal_DFs, axis=1)
# remove duplicate ID column
merged_DF=merged_DF.loc[:,~merged_DF.columns.duplicated()]
merged_DF.head(10)
#Remove Outliers
settings_DF.drop(settings_DF.loc[settings_DF['cut_no'].isin([17,94])].index, inplace=True)
merged_DF.drop(merged_DF.loc[merged_DF['cut_no'].isin([17,94])].index, inplace=True)

In [ ]:
# Based on Figure 6 in the DataSheet, we can set a wear treshold to 0.25
wear_threshold = 0.45
# Create a binary classification column based on the wear threshold
settings_DF['wear_class'] = np.where(settings_DF['VB'] > wear_threshold, 1, 0)
# 1 for wear, 0 for no wear

# Merge the settings_DF with the merged_DF on cut_no
binary_class_DF = pd.merge(merged_DF, settings_DF[['cut_no', 'wear_class']], on='cut_no', how='left')
# Plot the class distribution of wear_class
# Plot the distribution of wear classes (number of cuts per class)
plt.figure(figsize=(8, 5))
sns.countplot(data=settings_DF, x='wear_class')
plt.title('Distribution of Wear Classes (per Cut)')
plt.xlabel('Wear Class')
plt.ylabel('Number of Cuts')
plt.xticks([0, 1], ['No Wear', 'Wear'])
plt.show()

In [ ]:
# Define the function to reformat data
def reformat_data(selected_signals):
    global sequence_data, labels
    
    # Get the columns to keep
    columns_to_keep = ['cut_no', 'wear_class']
    for signal in selected_signals:
        columns_to_keep.extend([col for col in binary_class_DF.columns if signal in col])
    
    filtered_df = binary_class_DF[columns_to_keep]

    # Group the data by 'cut_no' to represent each time series
    new_sequence_data = []
    new_labels = []

    for cut_no, group in filtered_df.groupby('cut_no'):
        # Drop non-time-series columns and convert to numpy array
        time_series = group.drop(columns=['cut_no', 'wear_class']).to_numpy()
        new_sequence_data.append(time_series)
        new_labels.append(group['wear_class'].iloc[0])

    # Convert to numpy arrays
    sequence_data = np.array(new_sequence_data, dtype=object)
    labels = np.array(new_labels)

    print(f"Using signals: {selected_signals}")
    print(f"Number of sequences: {len(sequence_data)}")
    if len(sequence_data) > 0:
        print(f"Shape of first sequence: {sequence_data[0].shape}")

# Create an interactive widget that executes on user request
interact_manual(reformat_data, selected_signals=widgets.SelectMultiple(
    options=['vib_table', 'vib_spindle', 'AE_table', 'AE_spindle'],
    value=['vib_table', 'vib_spindle', 'AE_table', 'AE_spindle'],
    description='Signals',
    disabled=False
));

In [ ]:
from collections import Counter
# Get the length of each sequence in sequence_data
sequence_lengths = [len(seq) for seq in sequence_data]

# Use Counter to find the frequency of each length
length_counts = Counter(sequence_lengths)

# Check if all sequences have the same length
if len(length_counts) == 1:
    print("All sequences have the same length.")
else:
    print("Sequences have varying lengths.")
    # Optional: Display the distribution of lengths
    print("Length distribution:", length_counts)

# Get the most common length
most_common_length = length_counts.most_common(1)[0][0]

print(f"The most common sequence length is: {most_common_length}")

# Pad or truncate sequences to the most common length
padded_sequences = []
for seq in sequence_data:
    if len(seq) > most_common_length:
        # Truncate
        padded_sequences.append(seq[:most_common_length])
    elif len(seq) < most_common_length:
        # Pad with zeros
        padding = np.zeros((most_common_length - len(seq), seq.shape[1]))
        padded_sequences.append(np.vstack([seq, padding]))
    else:
        padded_sequences.append(seq)

# Update sequence_data with the padded/truncated sequences
sequence_data = np.array(padded_sequences)

print(f"All sequences have been reshaped to length {most_common_length}.")
print(f"New shape of sequence_data: {sequence_data.shape}")

## Model Setup

In [ ]:
# Convert filtered_labels to categorical format for multi-class classification
filtered_labels_categorical = to_categorical(labels)

# Convert filtered_sequences to numpy array for compatibility
filtered_sequences = np.array(sequence_data, dtype=object)

# Split the filtered data into training and testing sets
sequence_train, sequence_test, labels_train, labels_test = train_test_split(
    filtered_sequences, filtered_labels_categorical, test_size=0.2, random_state=42
)

# --- Refactor to Keras Functional API for Transfer Learning ---

# 1. Define the input layer
inputs = Input(shape=(most_common_length, sequence_train.shape[2]))

# 2. Define the "base model" - the feature extractor
# These layers will be frozen during transfer learning
base_layers = Conv1D(filters=32, kernel_size=5, activation='relu')(inputs)
base_layers = MaxPooling1D(pool_size=2)(base_layers)
base_layers = Conv1D(filters=64, kernel_size=5, activation='relu')(base_layers)
base_output = GlobalAveragePooling1D()(base_layers)

# For clarity, you can create a separate model for the base
base_model = Model(inputs=inputs, outputs=base_output, name="feature_extractor_base")
base_model.trainable = True # Set to True for initial training, False for transfer

# 3. Define the "head model" - the classifier
# This part will be replaced for the new task
head_layers = Dropout(0.5)(base_model.output)
head_layers = Dense(32, activation='relu')(head_layers)
outputs = Dense(filtered_labels_categorical.shape[1], activation='softmax')(head_layers)

# 4. Create the final model
model = Model(inputs=base_model.input, outputs=outputs)


# Compile the model
model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

# Model summary
model.summary()

## Model Train Setup

In [ ]:
# Convert sequence_train and sequence_test to proper numpy arrays with float dtype
X_train = np.array(sequence_train, dtype=np.float32)
X_test = np.array(sequence_test, dtype=np.float32)

In [ ]:
# Check for NaN or Inf values in X_train
print(f"Are there NaN values in X_train? {np.isnan(X_train).any()}")
print(f"Are there Inf values in X_train? {np.isinf(X_train).any()}")
# If invalid values are found, replace them
if np.isnan(X_train).any() or np.isinf(X_train).any():
    X_train = np.nan_to_num(X_train, nan=0.0, posinf=1e6, neginf=-1e6)
    print("Invalid values in X_train have been replaced.")
# Check for NaN or Inf values in labels_train
print(f"Are there NaN values in labels_train? {np.isnan(labels_train).any()}")
print(f"Are there Inf values in labels_train? {np.isinf(labels_train).any()}")

# If invalid values are found, replace them
if np.isnan(labels_train).any() or np.isinf(labels_train).any():
    labels_train = np.nan_to_num(labels_train, nan=0.0, posinf=1e6, neginf=-1e6)
    print("Invalid values in labels_train have been replaced.")
# Verify the shape of X_train and labels_train
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of labels_train: {labels_train.shape}")

# Verify the input shape expected by the model
print(f"Expected input shape: {model.input_shape}")

# Check for NaN or Inf values in X_train
print(f"Are there NaN values in X_train? {np.isnan(X_train).any()}")
print(f"Are there Inf values in X_train? {np.isinf(X_train).any()}")

# Check for NaN or Inf values in labels_train
print(f"Are there NaN values in labels_train? {np.isnan(labels_train).any()}")
print(f"Are there Inf values in labels_train? {np.isinf(labels_train).any()}")

In [ ]:
# Normalize X_train and X_test
# X_train = (X_train - np.mean(X_train, axis=0)) / np.std(X_train, axis=0)
# X_test = (X_test - np.mean(X_test, axis=0)) / np.std(X_test, axis=0)

In [ ]:
# More robust normalization to prevent data leakage
# Reshape data for scaler: from 3D to 2D
nsamples, nx, ny = X_train.shape
X_train_2d = X_train.reshape((nsamples * nx, ny))

scaler = StandardScaler()
# Fit scaler ONLY on the training data
scaler.fit(X_train_2d)

# Transform both train and test data
X_train_scaled_2d = scaler.transform(X_train_2d)
X_train = X_train_scaled_2d.reshape(nsamples, nx, ny)

nsamples_test, nx_test, ny_test = X_test.shape
X_test_2d = X_test.reshape((nsamples_test * nx_test, ny_test))
X_test_scaled_2d = scaler.transform(X_test_2d)
X_test = X_test_scaled_2d.reshape(nsamples_test, nx_test, ny_test)

print("Data has been scaled using a scaler fit on the training set.")

## Model Training

In [ ]:
# Train the model
history = model.fit(
    X_train,
    np.array(labels_train),
    epochs=400,
    batch_size=32,
    validation_split=0.2,
    verbose=2 #show loss at the end of each epoch
)

In [ ]:
# Evaluate the model
loss, accuracy = model.evaluate(X_test, np.array(labels_test))

# Create a figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# More detailed evaluation for confusion matrix
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(labels_test, axis=1)

# Plot Confusion matrix
cm = confusion_matrix(y_true_classes, y_pred_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues, ax=ax1)
ax1.set_title('Confusion Matrix')

# Plot loss and accuracy
ax2.plot(history.history['loss'], label='train loss')
ax2.plot(history.history['val_loss'], label='validation loss')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Loss')
ax2.set_title('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

# Print the evaluation results
print(f"Test Accuracy: {accuracy:.2f}")
print(f"Test Loss: {loss:.2f}")